# C389 CA1: try the trial model (deployment)

Loads the weights from `CA1_train_trial.ipynb` and serves the team's app: counts people on foot and wheeled crossers moving left and right, and draws the boxes.

Two ways to try photos:
- **Section 4:** upload photos straight into this notebook and see the result below the cell.
- **Section 5:** open the Gradio page through a public link (the same page used for the demo).

The app code comes from the team repo (`app/service.py`, `src/counting.py`), so what you try here is what ships.

**Before committing this notebook:** *Edit → Clear all outputs*. Outputs can hold photos of people.

> **Note (added 10 Nov):** saved from the team's Colab run on 7 Nov as deployment evidence: the final model loads on a T4 and the Gradio page opens with a public share link. The share link (expired) is hidden, and the photo-upload cell, which was cancelled, has its output removed. The code is the same as `notebooks/CA1_try_trial.ipynb`.

## 1. Settings

The repo is private, so cloning needs a GitHub token. Add one in Colab's **Secrets** panel (key icon on the left) named `GITHUB_TOKEN`, with notebook access switched on. A fine-grained token with read-only *Contents* on `c389-ca1` is enough. With no secret, the next cell asks for it.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/C389_Supernova_Model"
RUN_NAME   = "final_4cls_ig_v4_m"
WEIGHTS    = f"{DRIVE_ROOT}/runs/{RUN_NAME}/weights/best.pt"

REPO   = "Morn-Thuta/c389-ca1"
BRANCH = "feat/wheeled-classes"   # has the 4-class app code until the PRs are merged; then use "main"
ULTRALYTICS_VERSION = "8.4.120"

## 2. Install, get the code, connect Drive

In [ ]:
import os
import subprocess
from getpass import getpass

subprocess.run(["pip", "install", "-q", "ultralytics==" + ULTRALYTICS_VERSION, "gradio", "imagehash"], check=True)

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
if not token:
    token = getpass("GitHub token (read access to %s): " % REPO)

if not os.path.isdir("/content/c389-ca1"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                    "https://x-access-token:%s@github.com/%s.git" % (token, REPO),
                    "/content/c389-ca1"], check=True)
else:
    subprocess.run(["git", "-C", "/content/c389-ca1", "pull", "-q"], check=True)
del token
os.chdir("/content/c389-ca1")
print("code from branch", BRANCH)

from google.colab import drive
drive.mount("/content/drive")
if not os.path.isfile(WEIGHTS):
    raise FileNotFoundError(f"No weights at {WEIGHTS}. Run the training notebook first, or fix RUN_NAME.")
print("weights:", WEIGHTS)

code from branch feat/wheeled-classes
Mounted at /content/drive
weights: /content/drive/MyDrive/C389_Supernova_Model/runs/final_4cls_ig_v4_m/weights/best.pt


## 3. Load the model once

With the 4-class weights the mode reads "twist (people + wheeled)". The 2-class baseline reads "baseline (people only)".

In [ ]:
import logging
import sys

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", stream=sys.stdout, force=True)

from app import service
from app.app import build_demo
from src.config import CONF_THRESHOLD

DETECTOR = service.load_model(WEIGHTS)
print("mode   :", DETECTOR.mode)
print("classes:", DETECTOR.class_names)
print("default confidence threshold:", CONF_THRESHOLD)

2026-11-07 11:20:10,913 NumExpr defaulting to 2 threads.
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
2026-11-07 11:20:23,759 model loaded: /content/drive/MyDrive/C389_Supernova_Model/runs/final_4cls_ig_v4_m/weights/best.pt (twist (people + wheeled), input stretched to 512x512)
mode   : twist (people + wheeled), input stretched to 512x512
classes: ['person_left', 'person_right', 'wheeled_left', 'wheeled_right']
default confidence threshold: 0.4


## 4. Try your own photos here

Run the cell, click **Choose Files**, and pick one or more photos. Each one is shown with its boxes and counts.

- people: red = moving left, blue = moving right
- wheeled crossers (thick box): orange = moving left, purple = moving right
- Change `THRESHOLD` to see what a stricter or looser model finds.

In [ ]:
from google.colab import files
from IPython.display import display

THRESHOLD = CONF_THRESHOLD

uploaded = files.upload()
for name, data in uploaded.items():
    picture, reply = service.predict(data, THRESHOLD)
    print("=" * 60)
    print(name)
    if picture is None:
        print("refused:", reply["status"], reply["error"], "-", reply.get("detail", ""))
        continue
    counts = reply["counts"]
    print("people   left %(person_left)d  right %(person_right)d" % counts)
    print("wheeled  left %(wheeled_left)d  right %(wheeled_right)d" % counts)
    print("total    left %(total_left)d  right %(total_right)d" % counts)
    print("time: %.2f s" % reply["inference_seconds"])
    picture = picture.copy()
    picture.thumbnail((1200, 1200))
    display(picture)

## 5. Open the Gradio page (public link)

`share=True` prints a `gradio.live` link that anyone holding it can open, until this runtime stops (at most a week). Share it only with the team, and run section 6 when finished.

In [ ]:
demo = build_demo()
demo.launch(share=True)

2026-11-07 11:24:26,560 HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
2026-11-07 11:24:26,622 HTTP Request: GET http://127.0.0.1:7861/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-11-07 11:24:26,633 HTTP Request: HEAD http://127.0.0.1:7861/ "HTTP/1.1 200 OK"
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
2026-11-07 11:24:27,041 HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-11-07 11:24:27,368 HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
* Running on public URL: https://<link-hidden>.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
2026-11-07 11:24:28,941 HTTP Request: HEAD https://<link-hidden>.grad

## 6. Close the page

In [ ]:
demo.close()
print("The share link is closed.")